In [0]:
from pyspark.sql import functions as F

orders_silver = spark.table(
    "workspace.fashion_silver.orders"
)

customers_silver = spark.table(
    "workspace.fashion_silver.customers"
)

print("Customer analytics source tables loaded successfully.")

Customer analytics source tables loaded successfully.


In [0]:
customer_orders = (
    orders_silver
    .filter(
        (F.col("order_status") == "Delivered") &
        (F.col("customer_id").isNotNull())
    )
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "net_order_amount"
    )
)

print("Registered delivered orders:", customer_orders.count())
print(
    "Unique purchasing customers:",
    customer_orders.select("customer_id").distinct().count()
)

Registered delivered orders: 141899
Unique purchasing customers: 75733


In [0]:
analysis_date = customer_orders.agg(
    F.max("order_date").alias("max_date")
).first()["max_date"]

customer_rfm = (
    customer_orders
    .groupBy("customer_id")
    .agg(
        F.max("order_date").alias("last_purchase_date"),
        F.countDistinct("order_id").alias("frequency"),
        F.round(F.sum("net_order_amount"), 2).alias("monetary")
    )
    .withColumn(
        "recency",
        F.datediff(F.lit(analysis_date), F.col("last_purchase_date"))
    )
)

print("Analysis date:", analysis_date)
print("RFM customer records:", customer_rfm.count())

customer_rfm.show(10, truncate=False)

Analysis date: 2026-10-23
RFM customer records: 75733
+-----------+------------------+---------+---------+-------+
|customer_id|last_purchase_date|frequency|monetary |recency|
+-----------+------------------+---------+---------+-------+
|52826      |2026-10-06        |3        |38699.29 |17     |
|10619      |2025-11-26        |3        |43216.97 |331    |
|14851      |2025-01-21        |2        |60938.44 |640    |
|26350      |2026-08-12        |1        |2027.9   |72     |
|72658      |2025-12-25        |1        |7020.97  |302    |
|58092      |2026-10-18        |3        |24619.81 |5      |
|72819      |2026-01-02        |2        |44792.74 |294    |
|52181      |2026-04-20        |4        |27586.35 |186    |
|7288       |2026-09-13        |1        |16109.68 |40     |
|31178      |2025-09-10        |5        |120788.78|408    |
+-----------+------------------+---------+---------+-------+
only showing top 10 rows


In [0]:
from pyspark.sql.window import Window

recency_window = Window.orderBy(F.col("recency").asc())
frequency_window = Window.orderBy(F.col("frequency").desc())
monetary_window = Window.orderBy(F.col("monetary").desc())

customer_rfm = (
    customer_rfm
    .withColumn("R_score", 6 - F.ntile(5).over(recency_window))
    .withColumn("F_score", 6 - F.ntile(5).over(frequency_window))
    .withColumn("M_score", 6 - F.ntile(5).over(monetary_window))
    .withColumn(
        "RFM_score",
        F.concat(
            F.col("R_score").cast("string"),
            F.col("F_score").cast("string"),
            F.col("M_score").cast("string")
        )
    )
)

customer_rfm.select(
    "customer_id", "recency", "frequency", "monetary",
    "R_score", "F_score", "M_score", "RFM_score"
).show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+---------+-------+-------+-------+---------+
|customer_id|recency|frequency|monetary |R_score|F_score|M_score|RFM_score|
+-----------+-------+---------+---------+-------+-------+-------+---------+
|90151      |258    |8        |185718.52|4      |5      |5      |455      |
|89718      |218    |7        |166161.43|4      |5      |5      |455      |
|8716       |161    |8        |158690.06|4      |5      |5      |455      |
|6863       |43     |6        |156507.11|5      |5      |5      |555      |
|27395      |166    |7        |153469.72|4      |5      |5      |455      |
|694        |285    |7        |151106.88|4      |5      |5      |455      |
|94301      |235    |7        |148884.9 |4      |5      |5      |455      |
|64366      |1020   |5        |148525.61|1      |5      |5      |155      |
|95984      |294    |6        |146611.86|4      |5      |5      |455      |
|56248      |171    |6        |146210.39|4      |5      |5      |455      |
+-----------

In [0]:
customer_rfm = customer_rfm.withColumn(
    "customer_segment",
    F.when(
        (F.col("R_score") >= 4) &
        (F.col("F_score") >= 4) &
        (F.col("M_score") >= 4),
        "Champions"
    ).when(
        (F.col("R_score") >= 3) &
        (F.col("F_score") >= 4),
        "Loyal Customers"
    ).when(
        (F.col("R_score") >= 4) &
        (F.col("F_score") <= 2),
        "Recent Customers"
    ).when(
        (F.col("R_score") <= 2) &
        (F.col("F_score") >= 3),
        "At Risk"
    ).when(
        (F.col("R_score") <= 2) &
        (F.col("F_score") <= 2),
        "Hibernating"
    ).when(
        (F.col("R_score") >= 3) &
        (F.col("F_score") <= 2),
        "Potential Loyalists"
    ).otherwise("Other Customers")
)

customer_rfm.groupBy("customer_segment") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-----+
|customer_segment   |count|
+-------------------+-----+
|Hibernating        |19685|
|Champions          |14658|
|Loyal Customers    |12989|
|At Risk            |10607|
|Other Customers    |7187 |
|Potential Loyalists|6013 |
|Recent Customers   |4594 |
+-------------------+-----+



In [0]:
customer_analytics = (
    customer_rfm
    .join(
        customers_silver.select(
            "customer_id",
            "customer_name",
            "city",
            "gender",
            "age_group",
            "acquisition_channel",
            "preferred_language"
        ),
        on="customer_id",
        how="left"
    )
)

print("Customer analytics records:", customer_analytics.count())

customer_analytics.select(
    "customer_id",
    "city",
    "age_group",
    "acquisition_channel",
    "frequency",
    "monetary",
    "customer_segment"
).show(10, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer analytics records: 75733
+-----------+---------+---------+-------------------+---------+---------+----------------+
|customer_id|city     |age_group|acquisition_channel|frequency|monetary |customer_segment|
+-----------+---------+---------+-------------------+---------+---------+----------------+
|90151      |Mumbai   |25-34    |Facebook Ads       |8        |185718.52|Champions       |
|89718      |Bengaluru|35-44    |WhatsApp           |7        |166161.43|Champions       |
|8716       |Delhi    |45-54    |Facebook Ads       |8        |158690.06|Champions       |
|6863       |Pune     |25-34    |Organic Search     |6        |156507.11|Champions       |
|27395      |Mumbai   |45-54    |WhatsApp           |7        |153469.72|Champions       |
|694        |Pune     |35-44    |Organic Social     |7        |151106.88|Champions       |
|94301      |Delhi    |35-44    |Organic Search     |7        |148884.9 |Champions       |
|64366      |Pune     |55+      |Facebook Ads       |5  

In [0]:
city_segment_analysis = (
    customer_analytics
    .groupBy("city", "customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.round(F.avg("monetary"), 2).alias("avg_customer_spend"),
        F.round(F.avg("frequency"), 2).alias("avg_orders_per_customer")
    )
    .orderBy("city", F.desc("customer_count"))
)

city_segment_analysis.show(30, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------------+--------------+------------------+-----------------------+
|city     |customer_segment   |customer_count|avg_customer_spend|avg_orders_per_customer|
+---------+-------------------+--------------+------------------+-----------------------+
|Bengaluru|Hibernating        |4754          |14711.3           |1.0                    |
|Bengaluru|Champions          |3667          |50567.33          |3.04                   |
|Bengaluru|Loyal Customers    |3254          |29336.68          |2.51                   |
|Bengaluru|At Risk            |2619          |33363.08          |2.3                    |
|Bengaluru|Other Customers    |1762          |20496.79          |1.43                   |
|Bengaluru|Potential Loyalists|1525          |14614.32          |1.0                    |
|Bengaluru|Recent Customers   |1127          |14547.78          |1.0                    |
|Delhi    |Hibernating        |5033          |14591.78          |1.0                    |
|Delhi    

In [0]:
acquisition_analysis = (
    customer_analytics
    .groupBy("acquisition_channel", "customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.round(F.avg("monetary"), 2).alias("avg_customer_spend"),
        F.round(F.avg("frequency"), 2).alias("avg_orders_per_customer")
    )
    .orderBy("acquisition_channel", F.desc("customer_count"))
)

acquisition_analysis.show(50, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-------------------+--------------+------------------+-----------------------+
|acquisition_channel|customer_segment   |customer_count|avg_customer_spend|avg_orders_per_customer|
+-------------------+-------------------+--------------+------------------+-----------------------+
|Direct             |Hibernating        |2384          |14649.14          |1.0                    |
|Direct             |Champions          |1780          |50164.94          |2.98                   |
|Direct             |Loyal Customers    |1599          |29941.64          |2.55                   |
|Direct             |At Risk            |1340          |33061.14          |2.3                    |
|Direct             |Other Customers    |912           |21361.76          |1.43                   |
|Direct             |Potential Loyalists|813           |14093.78          |1.0                    |
|Direct             |Recent Customers   |560           |14700.11          |1.0                    |


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
segment_summary = (
    customer_analytics
    .groupBy("customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.round(F.avg("recency"), 2).alias("avg_recency_days"),
        F.round(F.avg("frequency"), 2).alias("avg_orders"),
        F.round(F.avg("monetary"), 2).alias("avg_customer_spend"),
        F.round(F.sum("monetary"), 2).alias("total_customer_revenue")
    )
    .orderBy(F.desc("total_customer_revenue"))
)

segment_summary.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+---------------+----------------+----------+------------------+----------------------+
|customer_segment   |total_customers|avg_recency_days|avg_orders|avg_customer_spend|total_customer_revenue|
+-------------------+---------------+----------------+----------+------------------+----------------------+
|Champions          |14653          |159.86          |3.02      |50724.76          |7.4326989893E8        |
|Loyal Customers    |12990          |295.62          |2.51      |29707.37          |3.8589872122E8        |
|At Risk            |10611          |844.71          |2.31      |33719.11          |3.5779351228E8        |
|Hibernating        |19681          |994.58          |1.0       |14765.16          |2.9059301882E8        |
|Other Customers    |7187           |270.1           |1.43      |21063.08          |1.5138032791E8        |
|Potential Loyalists|6014           |475.65          |1.0       |14507.05          |8.724542075E7         |
|Recent Customers   |4597   

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
recency_window = Window.orderBy(
    F.col("recency").asc(),
    F.col("customer_id").asc()
)

frequency_window = Window.orderBy(
    F.col("frequency").desc(),
    F.col("customer_id").asc()
)

monetary_window = Window.orderBy(
    F.col("monetary").desc(),
    F.col("customer_id").asc()
)

customer_rfm = (
    customer_rfm
    .withColumn("R_score", 6 - F.ntile(5).over(recency_window))
    .withColumn("F_score", 6 - F.ntile(5).over(frequency_window))
    .withColumn("M_score", 6 - F.ntile(5).over(monetary_window))
    .withColumn(
        "RFM_score",
        F.concat(
            F.col("R_score").cast("string"),
            F.col("F_score").cast("string"),
            F.col("M_score").cast("string")
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
customer_rfm = customer_rfm.withColumn(
    "customer_segment",
    F.when(
        (F.col("R_score") >= 4) &
        (F.col("F_score") >= 4) &
        (F.col("M_score") >= 4),
        "Champions"
    ).when(
        (F.col("R_score") >= 3) &
        (F.col("F_score") >= 4),
        "Loyal Customers"
    ).when(
        (F.col("R_score") >= 4) &
        (F.col("F_score") <= 2),
        "Recent Customers"
    ).when(
        (F.col("R_score") <= 2) &
        (F.col("F_score") >= 3),
        "At Risk"
    ).when(
        (F.col("R_score") <= 2) &
        (F.col("F_score") <= 2),
        "Hibernating"
    ).when(
        (F.col("R_score") >= 3) &
        (F.col("F_score") <= 2),
        "Potential Loyalists"
    ).otherwise("Other Customers")
)

customer_analytics = customer_rfm.join(
    customers_silver.select(
        "customer_id",
        "customer_name",
        "city",
        "gender",
        "age_group",
        "acquisition_channel",
        "preferred_language"
    ),
    on="customer_id",
    how="left"
)

print("Refreshed customer analytics:", customer_analytics.count())

customer_analytics.groupBy("customer_segment") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Refreshed customer analytics: 75733


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+-----+
|customer_segment   |count|
+-------------------+-----+
|Hibernating        |17335|
|At Risk            |12957|
|Champions          |12295|
|Loyal Customers    |11116|
|Other Customers    |9073 |
|Recent Customers   |7652 |
|Potential Loyalists|5305 |
+-------------------+-----+



In [0]:
segment_summary = (
    customer_analytics
    .groupBy("customer_segment")
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.round(F.avg("recency"), 2).alias("avg_recency_days"),
        F.round(F.avg("frequency"), 2).alias("avg_orders"),
        F.round(F.avg("monetary"), 2).alias("avg_customer_spend"),
        F.round(F.sum("monetary"), 2).alias("total_customer_revenue")
    )
    .orderBy(F.desc("total_customer_revenue"))
)

segment_summary.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+---------------+----------------+----------+------------------+----------------------+
|customer_segment   |total_customers|avg_recency_days|avg_orders|avg_customer_spend|total_customer_revenue|
+-------------------+---------------+----------------+----------+------------------+----------------------+
|Champions          |12295          |158.29          |3.22      |52628.54          |6.4706788041E8        |
|At Risk            |12957          |871.46          |2.07      |30249.76          |3.9194612547E8        |
|Loyal Customers    |11116          |348.47          |2.6       |32309.32          |3.5915041212E8        |
|Hibernating        |17335          |994.87          |1.0       |14793.49          |2.5644507488E8        |
|Other Customers    |9073           |275.07          |1.81      |26469.93          |2.4016170937E8        |
|Recent Customers   |7652           |176.12          |1.0       |14649.38          |1.1209702041E8        |
|Potential Loyalists|5305   

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
customer_analytics.select(
    F.count("*").alias("total_records"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum("monetary").alias("total_customer_revenue"),
    F.sum(F.when(F.col("city").isNull(), 1).otherwise(0)).alias("missing_city")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+----------------+----------------------+------------+
|total_records|unique_customers|total_customer_revenue|missing_city|
+-------------+----------------+----------------------+------------+
|        75733|           75733|  2.0838163396500049E9|           0|
+-------------+----------------+----------------------+------------+



In [0]:
registered_delivered_revenue = (
    orders_silver
    .filter(
        (F.col("order_status") == "Delivered") &
        F.col("customer_id").isNotNull()
    )
    .agg(F.round(F.sum("net_order_amount"), 2).alias("silver_revenue"))
)

gold_customer_revenue = customer_analytics.agg(
    F.round(F.sum("monetary"), 2).alias("gold_revenue")
)

registered_delivered_revenue.crossJoin(gold_customer_revenue).withColumn(
    "revenue_difference",
    F.round(F.col("silver_revenue") - F.col("gold_revenue"), 2)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+---------------+------------------+
| silver_revenue|   gold_revenue|revenue_difference|
+---------------+---------------+------------------+
|2.08381633965E9|2.08381633965E9|               0.0|
+---------------+---------------+------------------+



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.customer_analytics")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
segment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.fashion_gold.customer_segment_summary")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
for table_name in [
    "customer_analytics",
    "customer_segment_summary"
]:
    df = spark.table(f"workspace.fashion_gold.{table_name}")
    print(f"{table_name}: {df.count()} records")
    df.printSchema()

customer_analytics: 75733 records
root
 |-- customer_id: integer (nullable = true)
 |-- last_purchase_date: date (nullable = true)
 |-- frequency: long (nullable = true)
 |-- monetary: double (nullable = true)
 |-- recency: integer (nullable = true)
 |-- R_score: integer (nullable = true)
 |-- F_score: integer (nullable = true)
 |-- M_score: integer (nullable = true)
 |-- RFM_score: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age_group: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- preferred_language: string (nullable = true)

customer_segment_summary: 7 records
root
 |-- customer_segment: string (nullable = true)
 |-- total_customers: long (nullable = true)
 |-- avg_recency_days: double (nullable = true)
 |-- avg_orders: double (nullable = true)
 |-- avg_customer_spend: double (nullable = true)
 |-- 